# Specific conductance: implausible values and unit errors

**Why:** in the eligibility sensitivity review (2026-11-08, `mixed_model_results.ipynb` section 7b), one added dam, **MT-013**, made the before/after specific-conductance range 4× wider: its median went from 387 µS/cm before removal to **0.503 µS/cm** after. No river reads 0.5 µS/cm (that's close to distilled water); 0.503 looks like 503 µS/cm written in mS/cm. **NC-059** (286 → 43.5 µS/cm, 6 days after) is the next most extreme.

**What this notebook does (read-only; it changes nothing):**
1. MT-013's and NC-059's conductance records as reported: raw value, raw unit, organization, method, activity type, status. Is the unit wrong, and from when?
2. The whole cleaned table (`wq_measurement`): how many conductance values are implausibly low, and which stations / organizations / units they come from.
3. Stations whose conductance level jumps by ≥ 100× between years (the signature of a unit switch).
4. Which of those reach the analysis tables: main (3 / 2) and the sensitivity variants.

The result decides whether a cleaning rule is needed (applied to all data, main analysis included) and what it should be. Thresholds here are for screening, not rules.

**Update 2026-11-08:** the cleaning step now drops conductance below 2 µS/cm (`PLAUSIBLE_RANGES` in `etl/common/wq_rules.py`). After the cleaning rerun, sections 1–4 should show none of the ~0.5 values. Section 5 runs the same unit-slip screen on **every** measure; section 6 compares the rebuilt analysis tables with a backup taken before the rerun.

In [ ]:
from pathlib import Path
import os
import sys

import duckdb
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))
from etl.common.config import load_config
from etl.common.warehouse import analysis_path, warehouse_dir

DATA_ROOT = Path(os.environ.get("DAM_REMOVAL_PROJECT_ROOT", ROOT))
CASE_STUDY = "all_removals"
WAREHOUSE = warehouse_dir(load_config(CASE_STUDY, project_root=DATA_ROOT))
CHECK_DAMS = ["MT-013", "NC-059"]
LOW_US_CM = 5.0       # screening: below this is implausible for river water
JUMP_FACTOR = 100.0   # screening: year-to-year change in a station's median level

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

con = duckdb.connect()
con.execute(f"CREATE VIEW wq AS SELECT * FROM read_parquet('{WAREHOUSE / 'wq_measurement.parquet'}') "
            "WHERE parameter_std = 'specific_conductance'")
TABLES = {"main (3 / 2)": analysis_path(WAREHOUSE)}
for v in ("min1yr", "min1yr_2mo"):
    if analysis_path(WAREHOUSE, v).exists():
        TABLES[v] = analysis_path(WAREHOUSE, v)
for name, path in TABLES.items():
    con.execute(f"CREATE VIEW \"{name}\" AS SELECT * FROM read_parquet('{path}') "
                "WHERE parameter_std = 'specific_conductance'")
print(f"Warehouse: {WAREHOUSE}")
print(con.sql("SELECT count(*) AS conductance_results, count(DISTINCT station_id) AS stations FROM wq").df())

## 1. MT-013 and NC-059 as reported

Per dam, station, side and period: how many results, the median cleaned value, and the raw units, organizations and methods behind them. Then every result after removal. **What to look for:** a different organization, method or raw unit after removal, or raw values that are ~1000× smaller than before (mS/cm reported as µS/cm).

In [ ]:
src = "min1yr" if "min1yr" in TABLES else "main (3 / 2)"
dams = ", ".join(f"'{d}'" for d in CHECK_DAMS)
rows = con.sql(f"""
    SELECT a.AR_ID, a.station_id, a.side, a.period, a.sample_date, a.network_distance_km,
           w.result_value_raw, w.result_unit_raw, w.result_value_std, w.organization_id, w.provider,
           w.method_id, w.activity_type, w.result_status, w.value_type, w.qualifier_code, w.parameter
    FROM "{src}" a JOIN wq w USING (measurement_id)
    WHERE a.AR_ID IN ({dams})
    ORDER BY a.AR_ID, a.station_id, a.sample_date
""").df()
print(f"From {src}: {len(rows)} conductance results at {rows['station_id'].nunique()} stations")
summary = (rows.groupby(["AR_ID", "station_id", "side", "period"])
           .agg(results=("result_value_std", "size"), median_us_cm=("result_value_std", "median"),
                min_us_cm=("result_value_std", "min"), max_us_cm=("result_value_std", "max"),
                first=("sample_date", "min"), last=("sample_date", "max"),
                raw_units=("result_unit_raw", lambda s: sorted(s.dropna().unique())),
                organizations=("organization_id", lambda s: sorted(s.dropna().unique())),
                methods=("method_id", lambda s: sorted(s.dropna().unique())),
                activity_types=("activity_type", lambda s: sorted(s.dropna().unique())))
           .reset_index())
display(summary)
print("Every result after removal:")
display(rows[rows["period"] == "post"])

## 2. Implausibly low conductance in the whole cleaned table

Results below the screening value (5 µS/cm), by organization, raw unit and method. A cluster at one organization or one raw unit points to a reporting or unit error rather than real water.

In [ ]:
low = con.sql(f"""
    SELECT organization_id, provider, result_unit_raw, method_id,
           count(*) AS results, count(DISTINCT station_id) AS stations,
           median(result_value_std) AS median_us_cm, min(sample_date) AS first, max(sample_date) AS last
    FROM wq WHERE result_value_std < {LOW_US_CM} AND NOT is_censored
    GROUP BY ALL ORDER BY results DESC
""").df()
total = con.sql(f"SELECT count(*) FROM wq WHERE result_value_std < {LOW_US_CM} AND NOT is_censored").fetchone()[0]
print(f"Conductance results below {LOW_US_CM} µS/cm (not censored): {total:,}")
display(low.head(40))

## 3. Stations whose level jumps between years

Per station and year, the median conductance; flagged where it changes by ≥ 100× from the station's previous year with data. Real changes in a river's conductance are rarely that large; a unit switch (µS/cm ↔ mS/cm) is exactly 1000×.

In [ ]:
jumps = con.sql(f"""
    WITH yearly AS (
        SELECT station_id, year(sample_date) AS year, median(result_value_std) AS median_us_cm,
               count(*) AS results, string_agg(DISTINCT result_unit_raw, ', ') AS raw_units,
               string_agg(DISTINCT organization_id, ', ') AS organizations
        FROM wq WHERE NOT is_censored AND result_value_std > 0
        GROUP BY ALL
    ), stepped AS (
        SELECT *, lag(median_us_cm) OVER (PARTITION BY station_id ORDER BY year) AS prev_median,
                  lag(year) OVER (PARTITION BY station_id ORDER BY year) AS prev_year
        FROM yearly
    )
    SELECT station_id, prev_year, year, prev_median, median_us_cm, median_us_cm / prev_median AS ratio,
           results, raw_units, organizations
    FROM stepped
    WHERE prev_median IS NOT NULL
      AND (median_us_cm / prev_median >= {JUMP_FACTOR} OR prev_median / median_us_cm >= {JUMP_FACTOR})
    ORDER BY station_id, year
""").df()
print(f"Station-years with a ≥{JUMP_FACTOR:.0f}× step: {len(jumps)} at {jumps['station_id'].nunique()} stations")
display(jumps)

## 4. Do they reach the analysis tables?

For each analysis table: the dams and stations with low values or a jump, and how many conductance rows that is. **If the main (3 / 2) table has none, fixing this changes only the sensitivity results.**

In [ ]:
bad_stations = set(jumps["station_id"])
bad_low = set(con.sql(f"SELECT DISTINCT station_id FROM wq WHERE result_value_std < {LOW_US_CM} AND NOT is_censored").df()["station_id"])
reach = []
for name in TABLES:
    t = con.sql(f'SELECT AR_ID, station_id, period, result_value_std FROM "{name}"').df()
    for what, ids in (("low value station", bad_low), ("jump station", bad_stations)):
        hit = t[t["station_id"].isin(ids)]
        reach.append({"table": name, "screen": what, "dams": hit["AR_ID"].nunique(),
                      "stations": hit["station_id"].nunique(), "rows": len(hit),
                      "rows_below_screen": int((hit["result_value_std"] < LOW_US_CM).sum()),
                      "dam_ids": ", ".join(sorted(hit["AR_ID"].unique())[:15])})
display(pd.DataFrame(reach))

## 5. The same screen for every measure

Unit slips on ratio-scale measures (mg/L vs µg/L, mS/cm vs µS/cm) show up as a station's yearly median jumping by about 1,000×. This screen flags steps of ≥ 100× between a station's consecutive years with data, for every measure except water temperature and pH (not ratio scales: a °F/°C mix-up isn't a factor, and the plausible ranges already catch it). Discharge is included, but big real steps are possible there (a station moved, a dry year), so read its rows with that in mind. **What to look for:** many steps at one organization in the same year, or a step that reverses the next year.

In [ ]:
con.execute(f"CREATE OR REPLACE VIEW wq_all AS SELECT * FROM read_parquet('{WAREHOUSE / 'wq_measurement.parquet'}')")
NOT_RATIO = ("water_temp", "ph")
steps = con.sql(f"""
    WITH yearly AS (
        SELECT parameter_std, station_id, year(sample_date) AS year, median(result_value_std) AS median_value,
               count(*) AS results, string_agg(DISTINCT result_unit_raw, ', ') AS raw_units,
               string_agg(DISTINCT organization_id, ', ') AS organizations
        FROM wq_all
        WHERE NOT is_censored AND result_value_std > 0 AND parameter_std NOT IN {NOT_RATIO}
        GROUP BY ALL
    ), stepped AS (
        SELECT *, lag(median_value) OVER (PARTITION BY parameter_std, station_id ORDER BY year) AS prev_median,
                  lag(year) OVER (PARTITION BY parameter_std, station_id ORDER BY year) AS prev_year
        FROM yearly
    )
    SELECT parameter_std, station_id, prev_year, year, prev_median, median_value,
           median_value / prev_median AS ratio, results, raw_units, organizations
    FROM stepped
    WHERE prev_median IS NOT NULL
      AND (median_value / prev_median >= {JUMP_FACTOR} OR prev_median / median_value >= {JUMP_FACTOR})
    ORDER BY parameter_std, station_id, year
""").df()
print("Steps of >= 100x by measure:")
display(steps.groupby("parameter_std").agg(steps=("station_id", "size"), stations=("station_id", "nunique"),
                                          organizations=("organizations", lambda s: ", ".join(sorted(set(", ".join(s).split(", "))))[:150]))
        .sort_values("steps", ascending=False))
print("By measure, organization and year of the step (clusters suggest a reporting error):")
display(steps.groupby(["parameter_std", "organizations", "year"]).size().rename("steps")
        .sort_values(ascending=False).head(40).reset_index())

In [ ]:
# Which of the stepped stations reach the analysis tables, per measure?
step_keys = steps[["parameter_std", "station_id"]].drop_duplicates()
con.register("step_keys", step_keys)
reach_all = []
for name, path in TABLES.items():
    r = con.sql(f"""
        SELECT a.parameter_std, count(DISTINCT a.AR_ID) AS dams, count(DISTINCT a.station_id) AS stations,
               count(*) AS rows, string_agg(DISTINCT a.AR_ID, ', ') AS dam_ids
        FROM read_parquet('{path}') a JOIN step_keys k USING (parameter_std, station_id)
        GROUP BY ALL ORDER BY rows DESC
    """).df().assign(table=name)
    reach_all.append(r)
reach_all = pd.concat(reach_all, ignore_index=True)
display(reach_all[["table", "parameter_std", "dams", "stations", "rows", "dam_ids"]]
        .assign(dam_ids=lambda f: f["dam_ids"].str.slice(0, 120)))

## 6. After the rerun: what changed in the analysis tables?

Compares each rebuilt analysis table with the copy saved before the cleaning rerun (`data/processed/sc_floor_backup/`). **Expected:** only `specific_conductance` loses rows; every other measure has exactly the same measurements. Skipped when there's no backup.

In [ ]:
BACKUP = DATA_ROOT / "data" / "processed" / "sc_floor_backup"
for name, path in TABLES.items():
    old = BACKUP / path.name
    if not old.exists():
        print(f"{name}: no backup at {old}; skipped"); continue
    cmp = con.sql(f"""
        WITH o AS (SELECT parameter_std, measurement_id, AR_ID FROM read_parquet('{old}')),
             n AS (SELECT parameter_std, measurement_id, AR_ID FROM read_parquet('{path}'))
        SELECT coalesce(o.parameter_std, n.parameter_std) AS parameter_std,
               count(o.measurement_id) AS rows_before, count(n.measurement_id) AS rows_after,
               count(*) FILTER (WHERE n.measurement_id IS NULL) AS removed,
               count(*) FILTER (WHERE o.measurement_id IS NULL) AS added,
               count(DISTINCT o.AR_ID) AS dams_before, count(DISTINCT n.AR_ID) AS dams_after
        FROM o FULL JOIN n USING (measurement_id, AR_ID)
        GROUP BY ALL ORDER BY parameter_std
    """).df()
    changed = cmp[(cmp["removed"] > 0) | (cmp["added"] > 0)]
    print(f"\n{name}: measures with any change: {list(changed['parameter_std'])}")
    display(cmp)
    unexpected = changed[changed["parameter_std"] != "specific_conductance"]
    print("OK: only specific conductance changed" if unexpected.empty
          else f"CHECK: other measures changed: {list(unexpected['parameter_std'])}")

## 7. Targeted check of the other flagged cases (read-only)

Section 5 flagged ≥ 100× steps at stations that reach the analysis tables for measures other than conductance. Not every such step is an error: a dam removal can release a sediment pulse that raises turbidity or suspended sediment many-fold for a year or two, which is the effect under study. So these are inspected one by one; no rule is applied.

Per case: the dam and removal year; its stations for that measure (side, distance); each station's **full history** by year (count, median, min, max, raw units, organizations, methods, activity types), with years relative to removal; the most extreme individual readings with their raw values; and a physical cross-check:
- **TDS vs specific conductance** on the same station and day: TDS (mg/L) is normally about 0.55–0.75 × conductance (µS/cm). A ratio near 0.0007 or 700 points to a unit error (g/L or µg/L), not real water.
- **Turbidity and SSC / TSS vs discharge** on the same station and day: a real sediment pulse or storm shows high values with high flow, or clusters in the first years after removal; a unit error shows a constant factor over a whole period regardless of flow.

In [ ]:
CASES = [("WI-125", "tds"), ("WI-126", "tds"),
         ("MT-013", "turbidity"), ("PA-284", "turbidity"), ("VA-005", "turbidity"),
         ("WI-105", "turbidity"), ("WI-106", "turbidity"),
         ("CO-025", "ssc"), ("CO-025", "tss")]
loosest = TABLES.get("min1yr", TABLES["main (3 / 2)"])
case_dams = sorted({d for d, _ in CASES})
dam_info = con.sql(f"""SELECT AR_ID, Dam_Name, River, State, Year_Removed FROM read_parquet('{WAREHOUSE / 'dam_removal.parquet'}')
                       WHERE AR_ID IN ({", ".join(f"'{d}'" for d in case_dams)})""").df().set_index("AR_ID")
links = con.sql(f"""SELECT DISTINCT AR_ID, station_id, parameter_std, side, network_distance_km, removal_year
                    FROM read_parquet('{loosest}')
                    WHERE AR_ID IN ({", ".join(f"'{d}'" for d in case_dams)})""").df()
case_stations = sorted(set(links["station_id"]))
st_list = ", ".join(f"'{s}'" for s in case_stations)
hist = con.sql(f"""
    SELECT station_id, sample_date, year(sample_date) AS year, parameter_std, sample_fraction_std,
           result_value_raw, result_unit_raw, result_value_std, result_unit_std, is_censored,
           organization_id, provider, method_id, activity_type, result_status, value_type, qualifier_code
    FROM wq_all WHERE station_id IN ({st_list})
""").df()
print(f"{len(hist):,} results of any measure at the {len(case_stations)} case stations")
display(dam_info)

In [ ]:
def station_years(rows, removal_year):
    g = (rows.groupby(["station_id", "year"])
         .agg(n=("result_value_std", "size"), median=("result_value_std", "median"),
              min=("result_value_std", "min"), max=("result_value_std", "max"),
              raw_units=("result_unit_raw", lambda s: ", ".join(sorted(s.dropna().astype(str).unique()))),
              orgs=("organization_id", lambda s: ", ".join(sorted(s.dropna().unique()))),
              methods=("method_id", lambda s: ", ".join(sorted(s.dropna().astype(str).unique()))[:40]),
              activity=("activity_type", lambda s: ", ".join(sorted(s.dropna().unique()))[:40]),
              censored=("is_censored", "sum"))
         .reset_index())
    g.insert(2, "yrs_from_removal", g["year"] - removal_year)
    return g

def same_day(a, b, name_a, name_b):
    x = a.groupby(["station_id", "sample_date"])["result_value_std"].median().rename(name_a)
    y = b.groupby(["station_id", "sample_date"])["result_value_std"].median().rename(name_b)
    return pd.concat([x, y], axis=1, join="inner").reset_index()

for dam, param in CASES:
    info = dam_info.loc[dam] if dam in dam_info.index else None
    ry = int(info["Year_Removed"]) if info is not None and pd.notna(info["Year_Removed"]) else None
    lk = links[(links["AR_ID"] == dam) & (links["parameter_std"] == param)]
    print("\n" + "=" * 110)
    print(f"{dam} / {param}: {info['Dam_Name'] if info is not None else ''} ({info['River'] if info is not None else ''}, "
          f"{info['State'] if info is not None else ''}); removed {ry}")
    display(lk[["station_id", "side", "network_distance_km"]].sort_values(["side", "network_distance_km"]))
    rows = hist[(hist["parameter_std"] == param) & hist["station_id"].isin(lk["station_id"])]
    if rows.empty:
        print("  no results"); continue
    print("Station history by year (all years in the warehouse; the analysis window is removal year ±5, removal year excluded):")
    display(station_years(rows, ry if ry else 0))
    flagged = steps[(steps["parameter_std"] == param) & steps["station_id"].isin(lk["station_id"])]
    if len(flagged):
        print("Section 5 steps at these stations:")
        display(flagged[["station_id", "prev_year", "year", "prev_median", "median_value", "ratio", "raw_units", "organizations"]])
    ext = pd.concat([rows.nlargest(8, "result_value_std"), rows.nsmallest(5, "result_value_std")]).drop_duplicates()
    print("Most extreme readings (raw value as reported):")
    display(ext[["station_id", "sample_date", "result_value_raw", "result_unit_raw", "result_value_std", "is_censored",
                 "organization_id", "method_id", "activity_type", "result_status", "qualifier_code"]].sort_values("result_value_std"))
    if param == "tds":
        sc = hist[(hist["parameter_std"] == "specific_conductance") & hist["station_id"].isin(lk["station_id"])]
        pair = same_day(rows, sc, "tds_mg_l", "sc_us_cm")
        if len(pair):
            pair["ratio"] = pair["tds_mg_l"] / pair["sc_us_cm"]
            pair["year"] = pd.to_datetime(pair["sample_date"]).dt.year
            print("Cross-check: TDS / conductance on the same day (normal about 0.55-0.75):")
            display(pair.groupby(["station_id", "year"])["ratio"].describe()[["count", "min", "50%", "max"]].round(4))
        else:
            print("Cross-check: no same-day conductance")
    else:
        q = hist[(hist["parameter_std"] == "discharge") & hist["station_id"].isin(lk["station_id"])]
        pair = same_day(rows, q, param, "discharge_cfs")
        if len(pair):
            pair["year"] = pd.to_datetime(pair["sample_date"]).dt.year
            print(f"Cross-check: {param} vs same-day discharge (high values with high flow = plausible):")
            display(pair.groupby(["station_id", "year"]).agg(days=(param, "size"), median_value=(param, "median"),
                                                             max_value=(param, "max"), median_flow=("discharge_cfs", "median"),
                                                             max_flow=("discharge_cfs", "max")).round(2))
            print(f"  Spearman correlation {param} vs flow across all days: "
                  f"{pair[param].corr(pair['discharge_cfs'], method='spearman'):.2f} ({len(pair)} days)")
        else:
            print("Cross-check: no same-day discharge")

**Reading it.** A **reporting/unit error** looks like a constant factor (≈ 1000×, or the TDS/conductance ratio off by ≈ 1000) over a whole run of dates, often tied to one organization or a change of raw unit or method, with no link to flow and no link to the removal year. A **plausible environmental response** looks like high values on high-flow days, a rise concentrated in the first year or two after removal at downstream stations (sediment pulse), or isolated storm readings within the normal range for that river. Cases that fit neither stay as they are and are noted.

**Next step (decided after reading this, not here):** if the problem is confined to a recognizable pattern (an organization, a raw unit, a date), the fix belongs in `etl/clean_wq_results.py` / `etl/common/wq_rules.py` as a documented rule applied to every result: either convert (if the unit error is certain) or drop and count. Then rebuild the analysis tables and refit, and record the change in `docs/eligibility_sensitivity_prespec.md`. No dam is excluded by hand.